# Causal Inference Agentic Workflow

A LangGraph-based multi-agent system that answers causal questions using
Pearl's **causal ladder** (Association → Intervention → Counterfactual).

### Architecture

```
User Question
     │
     ▼
┌─────────────┐
│ Orchestrator │  ← classifies rung, extracts CausalQuery
└─────┬───────┘
      │  L1 / L2 / L3
      ▼
┌──────────────────────────┐
│ L1 Association Agent     │  P(Y|X)
│ L2 Intervention Agent    │  P(Y|do(X))
│ L3 Counterfactual Agent  │  P(Y_x | X=x', Y=y')
└──────────┬───────────────┘
           ▼
┌───────────────┐
│   Validator   │  ← identifiability, positivity, estimator check
└───────┬───────┘
   pass │ re_route → Orchestrator (loop)
   fail │
        ▼
┌───────────────┐
│  Synthesizer  │  ← final causal report
└───────────────┘
```

See `causal_inference_agentic_workflow.md` for the full design document.

In [ ]:
# Optional — uncomment to install dependencies
# %pip install langgraph openai anthropic python-dotenv

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from settings import Settings
from llm_client import build_llm_client
from graph import run_causal_workflow

print("Notebook directory:", notebook_dir)

## Configuration

API keys can come from the repo-root `.env` file **or** be set directly here.

```
# .env
LLM_PROVIDER=openai          # or "anthropic"
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_MODEL=gpt-4o-mini
ANTHROPIC_MODEL=claude-3-5-sonnet-latest
```

In [ ]:
# --- Override here if you prefer not to use .env ---
LLM_PROVIDER = None            # "openai" or "anthropic"; None → use .env
OPENAI_API_KEY_INLINE = None   # set to override .env
ANTHROPIC_API_KEY_INLINE = None

settings = Settings.from_env()
if LLM_PROVIDER:
    settings = Settings(
        llm_provider=LLM_PROVIDER,
        openai_api_key=OPENAI_API_KEY_INLINE or settings.openai_api_key,
        anthropic_api_key=ANTHROPIC_API_KEY_INLINE or settings.anthropic_api_key,
        openai_model=settings.openai_model,
        anthropic_model=settings.anthropic_model,
    )

llm = build_llm_client(settings)
print(f"LLM provider: {settings.llm_provider}")

## DAG & SCM Definitions

The agents reason over a shared DAG (and optionally an SCM for L3). Define one here
or use the defaults.

In [ ]:
# -- DAG for L1 / L2 examples --
smoking_dag = {
    "nodes": ["Smoking", "Tar", "Cancer", "Genetics"],
    "edges": [
        ["Smoking", "Tar"],
        ["Tar", "Cancer"],
        ["Genetics", "Smoking"],
        ["Genetics", "Cancer"],
    ],
    "description": (
        "Smoking → Tar → Cancer (front-door path). "
        "Genetics is an unmeasured common cause of Smoking and Cancer."
    ),
}

# -- SCM for L3 example --
drug_scm = {
    "structural_equations": {
        "X": "U_X",
        "Y": "0.6 * X + U_Y",
    },
    "exogenous": {
        "U_X": "Bernoulli(0.5)",
        "U_Y": "Normal(0, 0.1)",
    },
    "description": (
        "X = took drug (binary), Y = recovery score. "
        "Structural equation: Y = 0.6*X + U_Y."
    ),
}

drug_dag = {
    "nodes": ["X", "Y"],
    "edges": [["X", "Y"]],
    "description": "Simple RCT-like DAG: X → Y with no confounders.",
}

---

## Example 1 — L1 Association

> "Is smoking correlated with cancer after controlling for genetics?"

This is an **observational / associational** question → routed to the L1 agent.

In [ ]:
from IPython.display import Markdown

result_l1 = await run_causal_workflow(
    question="Is smoking correlated with cancer after controlling for genetics?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l1["ladder_rung"])
Markdown(result_l1["final_report"])

---

## Example 2 — L2 Intervention

> "What is the causal effect of smoking on cancer?"

This is an **interventional** question — P(Cancer | do(Smoking)) → routed to the L2 agent.
The DAG has an unmeasured confounder (Genetics) but a front-door path via Tar.

In [ ]:
result_l2 = await run_causal_workflow(
    question="What is the causal effect of smoking on cancer?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l2["ladder_rung"])
Markdown(result_l2["final_report"])

---

## Example 3 — L3 Counterfactual (with SCM)

> "A patient took the drug (X=1) and recovered (Y=0.7). What would their recovery have been had they NOT taken the drug?"

This is a **counterfactual** question → routed to L3. The SCM is fully specified.

In [ ]:
result_l3 = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=drug_scm,
)

print("Ladder rung:", result_l3["ladder_rung"])
Markdown(result_l3["final_report"])

---

## Example 4 — L3 Counterfactual (without SCM — graceful degradation)

Same counterfactual question, but **no SCM** is provided.
The L3 agent should degrade to bounds (Manski) and communicate the limitation.

In [ ]:
result_l3_no_scm = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=None,  # no SCM → agent should degrade to bounds
)

print("Ladder rung:", result_l3_no_scm["ladder_rung"])
Markdown(result_l3_no_scm["final_report"])

---

## Inspect Intermediate State

Each run returns the full `CausalState`. You can inspect every intermediate
artifact: the orchestrator's classification, the rung agent's analysis,
the validator's checks, and the final report.

In [ ]:
import json

print("=== L2 Example — Full State ===\n")
for key in ("ladder_rung", "causal_query", "analysis_result", "validation_result", "iteration"):
    print(f"── {key} ──")
    val = result_l2.get(key)
    if isinstance(val, dict):
        print(json.dumps(val, indent=2))
    else:
        print(val)
    print()